In [ ]:
#-------ACCESS TO GOOGLE DRIVE IN COLAB-------
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#------- BEFORE RUNING THE RECONSTRUCTION FOR NTH TIME CLEANING MEMORY -------

# cleans the memory occupied by the data (2D and 3D)
astra.data2d.clear()
astra.data3d.clear()

# cleans the memory occupied by the projectors (e.g., ray-driven, voxel-driven)
astra.projector.clear()

# cleans the memory occupied by the algorithms (e.g., FBP, SIRT, CGLS)
astra.algorithm.clear()

In [ ]:
!unzip /content/drive/MyDrive/CT/Proj2.zip -d ./dane

Archive:  /content/drive/MyDrive/CT/Proj2.zip
replace ./dane/Proj/img_00000.tif? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
# Instalation ASTRA Toolbox (if not already installed):
#!apt-get install python3-astra-toolbox

#if better version of ASTRA is needed, install it via pip (RECOMMENDED):
!pip install astra-toolbox

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 105.0 MB/s eta 0:00:00


In [ ]:
import os
import re
import numpy as np
import tifffile
import astra
from tqdm import tqdm
import matplotlib.pyplot as plt

In [ ]:

# =========================================================
# 1. Parameters from the XML file and derived parameters for ASTRA
# =========================================================
data_dir = "./dane/Proj"
output_dir = "./wyniki_4d"
os.makedirs(output_dir, exist_ok=True)

# Physical parameters from the XML file
voxel_size = 0.260449
sod_mm = 319.722
sdd_mm = 608.879

# Convert physical parameters to voxel units for ASTRA
DSO = sod_mm / voxel_size
DSD = sdd_mm / voxel_size

# Resolution (Z XML: X=189, Y=484, Z=189)
# ASTRA is typically used with (Z, Y, X) ordering for 3D volumes, so we will keep that in mind when defining the volume and projection geometries.
W_rec, H_rec = 189, 189    # width and height of the reconstructed volume in voxels (X and Y dimensions)
Z_slices = 484             # height of the reconstructed volume in voxels (Z dimension)

# Size of a single 2D projection (Radiogram)
W_proj = 189
H_proj = 484

In [ ]:
# =========================================================
# 2. PARAMETERS FOR THE RECONSTRUCTION
# =========================================================
num_frames = 16     # Dividing the 128 projections into 16 frames gives us 8 projections per frame, which is a very sparse sampling scenario.
iterations = 8      # Number of iterations per frame (I recommend starting with 5-10)
alpha = 0.4         # Low relaxation parameter compensating for noise (Eq. 2)

In [ ]:
# =========================================================
# 3. LOAD PROJECTIONS AND PREPARE DATA FOR ASTRA
# =========================================================
def natural_sort_key(s):
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r'(\d+)', s)]

all_files = sorted([f for f in os.listdir(data_dir) if f.lower().endswith((".tif", ".tiff"))], key=natural_sort_key)
selected_files = all_files[:128]

sample_img = tifffile.imread(os.path.join(data_dir, selected_files[0]))
H_proj, W_proj = sample_img.shape[:2]
print(f"Rzeczywisty rozmiar radiogramu: {W_proj}x{H_proj} pikseli")
print(f"Wczytywanie {len(selected_files)} zdjęć do pamięci...")

p_measured_all = np.zeros((H_proj, len(selected_files), W_proj), dtype=np.float32)
all_angles_rad = []

for idx, filename in enumerate(tqdm(selected_files)):
    img = tifffile.imread(os.path.join(data_dir, filename)).astype(np.float32)

    # 1. Fixing potential zero values to avoid division by zero in the log transform
    I0 = np.quantile(img, 0.995)
    img_clipped = np.clip(img, a_min=1e-3, a_max=I0)

    # Now I0 / img_clipped zawsze >= 1, so img_log jest zawsze dodatni (>=0)
    img_log = np.log(I0 / img_clipped)

    p_measured_all[:, idx, :] = img_log

    theta = -2 * np.pi * idx / len(selected_files)
    all_angles_rad.append(theta)

all_angles_rad = np.array(all_angles_rad, dtype=np.float32)

Rzeczywisty rozmiar radiogramu: 190x504 pikseli
Wczytywanie 128 zdjęć do pamięci...


100%|██████████| 128/128 [00:00<00:00, 580.28it/s]


In [ ]:
# =========================================================
# 4. PRIOR KNOWLEDGE (TRUE BACKGROUND) AND WEIGHT MATRIX
# =========================================================
vol_geom = astra.create_vol_geom(W_rec, H_rec, Z_slices)

# Detector magnification parameter
det_spacing = DSD / DSO

print("\nKROK 1: Generowanie wiedzy a priori (SIRT ze wszystkich 128 rzutów)...")
print("Algorytm uczy się wyglądu probówki i gliceryny (to potrwa kilkanaście sekund).")

# Geometry for the full set of projections (128 angles) - needed for the initial SIRT reconstruction to get the true background
proj_geom_all = astra.create_proj_geom('cone', det_spacing, det_spacing, H_proj, W_proj, all_angles_rad, DSO, (DSD - DSO))

# Configuration for the fast SIRT algorithm in ASTRA
cfg = astra.astra_dict('SIRT3D_CUDA')
cfg['ProjectionDataId'] = astra.data3d.create('-sino', proj_geom_all, p_measured_all)
cfg['ReconstructionDataId'] = astra.data3d.create('-vol', vol_geom, data=0.0)
cfg['option'] = {'MinConstraint': 0.0}

alg_id = astra.algorithm.create(cfg)
astra.algorithm.run(alg_id, 15) # 15 iterations for the initial SIRT reconstruction to get the true background
initial_volume = astra.data3d.get(cfg['ReconstructionDataId'])

# Cleaning up ASTRA memory after the initial SIRT reconstruction
astra.algorithm.delete(alg_id)
astra.data3d.delete(cfg['ProjectionDataId'])
astra.data3d.delete(cfg['ReconstructionDataId'])

'''
# =================================================================================
# Creating weight matrix with prior knowledge and assumed dynamical area
# =================================================================================
print("Tworzenie rozszerzonej macierzy wag...")
#nadajemu całemu tłu wage 10%
weight_volume = np.ones((Z_slices, H_rec, W_rec), dtype=np.float32) * 0.1
center_x, center_y = W_rec // 2, H_rec // 2
y_grid, x_grid = np.ogrid[:H_rec, :W_rec]
mask = (x_grid - center_x)**2 + (y_grid - center_y)**2 <= 80**2
weight_volume[:, mask] = 1.0
'''

# ===========================================================================================
# Creating weight matrix based on physical geometry and partial volume effect considerations
# ===========================================================================================
print("Tworzenie macierzy wag na podstawie geometrii fizycznej...")

# 1. Weight volume initialization with default low weight (0.1) for the background, including the tube walls
weight_volume = np.ones((Z_slices, H_rec, W_rec), dtype=np.float32) * 0.1

# 2. Physical parameters from the scan metadata (XML file) - we will use these to define the region of interest (ROI) for the mask
inner_diameter_mm = 21.0
inner_radius_mm = inner_diameter_mm / 2.0
voxel_size_mm = 0.260449  # From the scanner metadata (X/Y/Z axis)

# 3. Analitical calculation of the mask radius in pixels
exact_radius_px = inner_radius_mm / voxel_size_mm

# 4. Margin for Partial Volume Effect (-2 pixels)
safe_radius_px = int(exact_radius_px) - 2

print(f"-> Fizyczny promień rury: {inner_radius_mm:.2f} mm")
print(f"-> Rozmiar woksela: {voxel_size_mm:.6f} mm/px")
print(f"-> Wyliczony bezpieczny promień maski (ROI): {safe_radius_px} pikseli")

# 5. Generating the circular mask based on the calculated safe radius and applying it to the weight volume
center_x, center_y = W_rec // 2, H_rec // 2
center_x = 84
center_y = 90
y_grid, x_grid = np.ogrid[:H_rec, :W_rec]
print("100% safe radius", safe_radius_px)
safe_radius_px=safe_radius_px*0.5
mask = (x_grid - center_x)**2 + (y_grid - center_y)**2 <= safe_radius_px**2
weight_volume[:, mask] = 1.0

print("xenter_x", center_x, "center_y", center_y, "safe_radius", safe_radius_px)


KROK 1: Generowanie wiedzy a priori (SIRT ze wszystkich 128 rzutów)...
Algorytm uczy się wyglądu probówki i gliceryny (to potrwa kilkanaście sekund).
Tworzenie macierzy wag na podstawie geometrii fizycznej...
-> Fizyczny promień rury: 10.50 mm
-> Rozmiar woksela: 0.260449 mm/px
-> Wyliczony bezpieczny promień maski (ROI): 38 pikseli
100% safe radius 38
xenter_x 84 center_y 90 safe_radius 19.0


In [ ]:
# =========================================================
# 5. Algorithm 4D SIRT/WBP WITH STABILIZERS FROM THE PUBLICATION (OPTIMIZED AND SAFE) 
# PUBLICATION: Improving image quality in fast, time‑resolved micro‑CT by weighted back projection
# Marjolein Heyndrickx1, Tom Bultreys2, Wannes Goethals1, Luc Van Hoorebeke1 & Matthieu N. Boone1
# doi.org/10.1038/s41598-020-74827-x
# =========================================================

projections_per_frame = len(selected_files) // num_frames
det_spacing = DSD / DSO

print(f"\nRozpoczynam pętlę iteracyjną 4D. Liczba klatek: {num_frames}")

for frame_idx in range(num_frames):
    start_idx = frame_idx * projections_per_frame
    end_idx = start_idx + projections_per_frame

    angles_subset = all_angles_rad[start_idx:end_idx]
    p_subset = p_measured_all[:, start_idx:end_idx, :]

    print(f"\n=== Klatka {frame_idx+1}/{num_frames} ({len(angles_subset)} rzutów czasowych) ===")
    proj_geom = astra.create_proj_geom('cone', det_spacing, det_spacing, H_proj, W_proj, angles_subset, DSO, (DSD - DSO))

    # --- Stabilizators SIRT/WBP ---
    # a) Sum of weights along the ray (Denominator from Eq. 3 in the publication)
    _, sum_weights_proj = astra.create_sino3d_gpu(weight_volume, proj_geom, vol_geom)

    # b) Col sum: number of rays through each voxel (stabilizes the iteration)
    ones_proj = np.ones_like(p_subset)
    _, col_sum = astra.create_backprojection3d_gpu(ones_proj, proj_geom, vol_geom)

    volume_k = initial_volume.copy()

    for it in range(iterations):
        # 1. Simulate projections from the current volume estimate
        _, q_simulated = astra.create_sino3d_gpu(volume_k, proj_geom, vol_geom)

        # 2. Measured error
        error = p_subset - q_simulated

        # 3. Divide by the sum of weights 
        normalized_error = error / (sum_weights_proj + 1e-8)

        # 4. Back-project the normalized error
        _, back_projected_error = astra.create_backprojection3d_gpu(normalized_error, proj_geom, vol_geom)

        # 5. Actualization WBP
        stable_update = back_projected_error / (col_sum + 1e-8)
        volume_k = volume_k + alpha * weight_volume * stable_update

        # Physical constraint: Densities cannot be negative
        volume_k[volume_k < 0] = 0

        # Reporting values
        print(f"  Iter {it+1}/{iterations} | Max_Gęstość: {volume_k.max():.6f} | Min_Gęstość: {volume_k.min():.6f}")

    output_filename = os.path.join(output_dir, f"frame_{frame_idx:03d}.tif")
    tifffile.imwrite(output_filename, volume_k, imagej=True, metadata={'axes': 'ZYX'})
    print(f"-> Zapisano gotową klatkę: {output_filename}")


Rozpoczynam pętlę iteracyjną 4D. Liczba klatek: 16

=== Klatka 1/16 (8 rzutów czasowych) ===
  Iter 1/8 | Max_Gęstość: 0.051964 | Min_Gęstość: 0.000000
  Iter 2/8 | Max_Gęstość: 0.051964 | Min_Gęstość: 0.000000
  Iter 3/8 | Max_Gęstość: 0.051964 | Min_Gęstość: 0.000000
  Iter 4/8 | Max_Gęstość: 0.051964 | Min_Gęstość: 0.000000
  Iter 5/8 | Max_Gęstość: 0.051964 | Min_Gęstość: 0.000000
  Iter 6/8 | Max_Gęstość: 0.051964 | Min_Gęstość: 0.000000
  Iter 7/8 | Max_Gęstość: 0.051964 | Min_Gęstość: 0.000000
  Iter 8/8 | Max_Gęstość: 0.051964 | Min_Gęstość: 0.000000
-> Zapisano gotową klatkę: ./wyniki_4d/frame_000.tif

=== Klatka 2/16 (8 rzutów czasowych) ===
  Iter 1/8 | Max_Gęstość: 0.050022 | Min_Gęstość: 0.000000
  Iter 2/8 | Max_Gęstość: 0.048190 | Min_Gęstość: 0.000000
  Iter 3/8 | Max_Gęstość: 0.047981 | Min_Gęstość: 0.000000
  Iter 4/8 | Max_Gęstość: 0.053510 | Min_Gęstość: 0.000000
  Iter 5/8 | Max_Gęstość: 0.058414 | Min_Gęstość: 0.000000
  Iter 6/8 | Max_Gęstość: 0.062810 | Min_Gęs

In [ ]:
# 1. Command to zip the results folder (wyniki_4d) into a single file (moj_film_4d.zip)
!zip -r moj_film_4d.zip wyniki_4d/

# 2. Command to open the download window in the browser
from google.colab import files
files.download('moj_film_4d.zip')

updating: wyniki_4d/ (stored 0%)
updating: wyniki_4d/frame_002.tif (deflated 32%)
updating: wyniki_4d/frame_011.tif (deflated 22%)
updating: wyniki_4d/frame_015.tif (deflated 31%)
updating: wyniki_4d/frame_009.tif (deflated 19%)
updating: wyniki_4d/frame_014.tif (deflated 26%)
updating: wyniki_4d/frame_013.tif (deflated 24%)
updating: wyniki_4d/frame_001.tif (deflated 32%)
updating: wyniki_4d/frame_007.tif (deflated 23%)
updating: wyniki_4d/frame_004.tif (deflated 32%)
updating: wyniki_4d/frame_003.tif (deflated 35%)
updating: wyniki_4d/frame_005.tif (deflated 27%)
updating: wyniki_4d/frame_006.tif (deflated 24%)
updating: wyniki_4d/frame_008.tif (deflated 22%)
updating: wyniki_4d/frame_010.tif (deflated 19%)
updating: wyniki_4d/frame_000.tif (deflated 34%)
updating: wyniki_4d/frame_012.tif (deflated 24%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>